# Traffic forecasting benchmark (V19): node-level spatio-temporal transformer

Runs the **standard METR-LA / PEMS-BAY protocol** (12 -> 12 steps, DCRNN 70/10/20 split,
masked MAE / RMSE / MAPE on raw mph, horizons 15 / 30 / 60 min) for:

* `st_transformer`: new node-level spatio-temporal transformer (`traffic_transformer/st_model.py`)
* `st_transformer --graph-bias`: plus the road-graph hop-distance attention prior
* `legacy`: the V18 `TrafficTransformer`, evaluated under the same protocol

Every run writes to `results/benchmark/<run-name>/` **on Drive** (`train_log.txt`, `history.json`,
`results.json`, `best.pt`, `last.pt`). Re-running the notebook skips finished runs and resumes
interrupted ones from `last.pt`.

In [ ]:
# === 1. Setup: mount Drive, copy code + data to the local SSD ===
import os, shutil, subprocess, sys, json, time
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction/Transformer_Versions'
V19 = f'{DRIVE_BASE}/COLAB_NOBASELINE_V19'
DATA_SRC = f'{DRIVE_BASE}/COLAB_NOBASELINE_V18/Transformers_Input'
OUT = f'{V19}/results/benchmark'          # on Drive: survives disconnects, monitorable
LOCAL = '/content/v19'
LOCAL_DATA = '/content/data'

if os.path.exists(LOCAL):
    shutil.rmtree(LOCAL)
shutil.copytree(V19, LOCAL, ignore=shutil.ignore_patterns('results', '*.ipynb'))
os.makedirs(LOCAL_DATA, exist_ok=True)
for f in ['PEMS-BAY.csv', 'METR-LA.csv', 'adj_PEMS-BAY.pkl', 'adj_METR-LA.pkl',
          'weather_PEMS-BAY_era5_local.csv', 'weather_METR-LA_era5_local.csv',
          'clean_weather_data_pems_bay.csv', 'clean_weather_data_metr_la.csv']:
    if not os.path.exists(f'{LOCAL_DATA}/{f}'):
        shutil.copy2(f'{DATA_SRC}/{f}', f'{LOCAL_DATA}/{f}')
os.makedirs(OUT, exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'holidays', 'seaborn', 'pytz', 'pyyaml'])

import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'
info = {'torch': torch.__version__, 'gpu': gpu,
        'mem_gb': round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1) if gpu != 'none' else 0,
        'started': time.strftime('%Y-%m-%d %H:%M:%S')}
print(info)
with open(f'{OUT}/colab_session.json', 'w') as f:
    json.dump(info, f)

In [ ]:
# === 2. Experiment queue ===
# (run_name, extra CLI args). Fixed names so re-runs resume instead of restarting.
EXPERIMENTS = [
    ('PEMS-BAY_st_base',    '--dataset PEMS-BAY --model st_transformer'),
    ('PEMS-BAY_st_graph',   '--dataset PEMS-BAY --model st_transformer --graph-bias'),
    ('METR-LA_st_base',     '--dataset METR-LA --model st_transformer'),
    ('METR-LA_st_graph',    '--dataset METR-LA --model st_transformer --graph-bias'),
    ('METR-LA_legacy',      '--dataset METR-LA --model legacy --batch-size 32'),
]
MAX_CONCURRENT = 4          # runs sharing the GPU at once
COMMON = '--precision tf32 --save-predictions'

In [ ]:
# === 3. Launch + monitor (keeps the runtime busy until the queue is done) ===
import collections

def is_done(name):
    return os.path.exists(f'{OUT}/{name}/results.json')

def last_line(name):
    p = f'{OUT}/{name}/train_log.txt'
    if not os.path.exists(p):
        return '(starting)'
    with open(p, encoding='utf-8', errors='ignore') as f:
        lines = [l.rstrip() for l in f if l.strip()]
    ep = [l for l in lines if l.startswith('Epoch')]
    return (ep[-1] if ep else lines[-1])[:150] if lines else '(empty)'

queue = collections.deque([(n, a) for n, a in EXPERIMENTS if not is_done(n)])
running = {}
print(f'{len(queue)} runs queued, {len(EXPERIMENTS) - len(queue)} already done')
t_start = time.time()
while queue or running:
    while queue and len(running) < MAX_CONCURRENT:
        name, args = queue.popleft()
        resume = '--resume' if os.path.exists(f'{OUT}/{name}/last.pt') else ''
        cmd = (f'cd {LOCAL} && {sys.executable} -W ignore run_benchmark.py {args} {COMMON} {resume} '
               f'--input-dir {LOCAL_DATA} --output-dir "{OUT}" --run-name {name}')
        os.makedirs(f'{OUT}/{name}', exist_ok=True)
        logf = open(f'{OUT}/{name}/stdout.txt', 'a')
        running[name] = (subprocess.Popen(cmd, shell=True, stdout=logf, stderr=subprocess.STDOUT), logf)
        print(f'[launch] {name} {resume}')
        time.sleep(20)
    time.sleep(120)
    for name in list(running):
        proc, logf = running[name]
        if proc.poll() is not None:
            logf.close()
            status = 'OK' if proc.returncode == 0 else f'FAILED (exit {proc.returncode})'
            print(f'[finish] {name}: {status}')
            del running[name]
    elapsed = (time.time() - t_start) / 60
    print(f'--- {time.strftime("%H:%M:%S")} | {elapsed:.0f} min | running {len(running)}, queued {len(queue)}')
    for name in running:
        print(f'   {name:22s} {last_line(name)}')
    status = {'time': time.strftime('%Y-%m-%d %H:%M:%S'), 'running': list(running),
              'queued': [q[0] for q in queue], 'elapsed_min': round(elapsed, 1)}
    with open(f'{OUT}/queue_status.json', 'w') as f:
        json.dump(status, f)
print('All runs finished.')

In [ ]:
# === 4. Results table ===
import glob
import pandas as pd
rows = []
for p in sorted(glob.glob(f'{OUT}/*/results.json')):
    r = json.load(open(p))
    if 'test' not in r:
        continue
    t = r['test']
    row = {'run': os.path.basename(os.path.dirname(p)), 'params': r.get('params'),
           'best_epoch': r.get('best_epoch'), 'min/epoch': round((r.get('mean_epoch_time_s') or 0) / 60, 2)}
    for h in ['h3', 'h6', 'h12', 'all']:
        for m in ['mae', 'rmse', 'mape']:
            row[f'{h}_{m}'] = round(t[h][m], 3)
    rows.append(row)
df = pd.DataFrame(rows)
pd.set_option('display.width', 250)
print(df.to_string(index=False))
df.to_csv(f'{OUT}/summary.csv', index=False)

In [ ]:
!cd /content/v19 && python -W ignore queue_runner.py --queue "/content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction/Transformer_Versions/COLAB_NOBASELINE_V19/results/queue.json" --out "/content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction/Transformer_Versions/COLAB_NOBASELINE_V19/results/benchmark" --data /content/data --max-concurrent 4 --busy-minutes 0 --common=--save-predictions